# EU AI Act — Natural Language Interface

A retrieval-augmented interface for asking questions about AI projects in the EU,
answered from Regulation (EU) 2024/1689.

This notebook covers the **embedding stage**: load the parsed corpus, embed each
item's `embed_text`, and cache the result so the cost is paid once.

| Artefact | File |
|---|---|
| corpus | `data/JSON/eu_ai_act.json` |
| vectors | `data/Embeddings/eu_ai_act_embeddings.npy` |
| row → item id | `data/Embeddings/embedding_ids.json` |
| run metadata | `data/Embeddings/embedding_metadata.json` |

**Caching.** Every run builds a metadata record describing what *would* be produced,
including a SHA-256 fingerprint over every `(id, embed_text)` pair. If that record
matches the one saved beside the vectors, the vectors are loaded from disk. If
anything meaningful differs — the model, the field embedded, normalisation, the item
count, or a single character of any item's text — the embeddings are rebuilt.


## 1. Setup

Configuration lives in one place so a change to any of it invalidates the cache.


In [1]:
import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import sentence_transformers
from sentence_transformers import SentenceTransformer


def find_project_root(start: Path) -> Path:
    """Find the repo root by looking for data/JSON, so this runs from anywhere."""
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "JSON").is_dir():
            return candidate
    raise FileNotFoundError(f"No data/JSON directory found above {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
JSON_PATH = PROJECT_ROOT / "data" / "JSON" / "eu_ai_act.json"
EMBED_DIR = PROJECT_ROOT / "data" / "Embeddings"
VECTORS_PATH = EMBED_DIR / "eu_ai_act_embeddings.npy"
IDS_PATH = EMBED_DIR / "embedding_ids.json"
METADATA_PATH = EMBED_DIR / "embedding_metadata.json"

# --- Embedding configuration -------------------------------------------------
MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBED_FIELD = "embed_text"      # the field whose text becomes a vector
NORMALIZE = True                # unit vectors, so cosine similarity is a dot product
DTYPE = "float32"
BATCH_SIZE = 32

# BGE models expect short queries to carry a retrieval instruction, while the
# passages themselves are embedded bare. Used at query time, not here.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("project root :", PROJECT_ROOT)
print("model        :", MODEL_NAME)
print("device       :", DEVICE)
print("torch        :", torch.__version__)
print("sentence-tf  :", sentence_transformers.__version__)


c:\Users\rojus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


project root : C:\Users\rojus\OneDrive\Documents\AI Engineering\Job Application Stuff\EY Task\Natural-Language-Interface-for-Eur-Lex-Regulation
model        : BAAI/bge-small-en-v1.5
device       : cpu
torch        : 2.14.0+cpu
sentence-tf  : 6.0.1


## 2. Load the corpus


In [2]:
corpus = json.loads(JSON_PATH.read_text(encoding="utf-8"))

missing = [i["id"] for i in corpus if not i.get(EMBED_FIELD, "").strip()]
if missing:
    raise ValueError(f"{len(missing)} items have no {EMBED_FIELD}, e.g. {missing[:5]}")

print(f"{len(corpus)} items loaded from {JSON_PATH.name}")
for kind in ("article", "definition", "recital", "annex"):
    print(f"  {kind:11} {sum(1 for i in corpus if i['type'] == kind):4}")


980 items loaded from eu_ai_act.json
  article      578
  definition    68
  recital      193
  annex        141


## 3. Describing the run

The fingerprint is the heart of the cache. It hashes every item's id **and** its
embedding text in order, so it changes if an item is added, removed, reordered,
renamed, or edited — any of which would leave the saved vectors misaligned with the
corpus.

`CACHE_KEYS` names the fields that must match for cached vectors to be reusable.
Everything else in the metadata is recorded for traceability but does not force a
rebuild: a newer torch does not change what this model produces.


In [3]:
# Fields that must match for the cached vectors to be considered valid.
CACHE_KEYS = ("model_name", "embedded_field", "normalized", "dtype",
              "item_count", "corpus_fingerprint")


def corpus_fingerprint(items: list[dict]) -> str:
    """SHA-256 over every (id, embed_text) pair, in order."""
    digest = hashlib.sha256()
    for item in items:
        digest.update(item["id"].encode("utf-8"))
        digest.update(b"\x00")
        digest.update(item[EMBED_FIELD].encode("utf-8"))
        digest.update(b"\x00")
    return digest.hexdigest()


def build_run_metadata(items: list[dict]) -> dict:
    """Describe what this run would produce, before producing it."""
    return {
        # --- cache keys ---
        "model_name": MODEL_NAME,
        "embedded_field": EMBED_FIELD,
        "normalized": NORMALIZE,
        "dtype": DTYPE,
        "item_count": len(items),
        "corpus_fingerprint": corpus_fingerprint(items),
        # --- traceability only ---
        "source_json": JSON_PATH.name,
        "device": DEVICE,
        "torch_version": torch.__version__,
        "sentence_transformers_version": sentence_transformers.__version__,
    }


run_metadata = build_run_metadata(corpus)
print("fingerprint:", run_metadata["corpus_fingerprint"][:32], "...")
print("items      :", run_metadata["item_count"])


fingerprint: 0f01b24dd40867f2fc01088477d0b118 ...
items      : 980


## 4. Cache check

Returns the reasons a rebuild is needed, rather than a bare boolean, so a rebuild
always explains itself.


In [4]:
def cache_status(current: dict) -> tuple[bool, list[str]]:
    """Is the cache on disk usable for this run? If not, why not."""
    reasons = []

    for path, label in ((VECTORS_PATH, "vectors"), (IDS_PATH, "ids"),
                        (METADATA_PATH, "metadata")):
        if not path.exists():
            reasons.append(f"no {label} file at {path.name}")
    if reasons:
        return False, reasons

    try:
        saved = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return False, [f"metadata file is unreadable ({exc})"]

    for key in CACHE_KEYS:
        old, new = saved.get(key), current[key]
        if old != new:
            if key == "corpus_fingerprint":
                reasons.append("corpus changed (ids or text differ from the saved run)")
            else:
                reasons.append(f"{key} changed: {old!r} -> {new!r}")

    # The files must also agree with each other.
    if not reasons:
        saved_ids = json.loads(IDS_PATH.read_text(encoding="utf-8"))
        rows = np.load(VECTORS_PATH, mmap_mode="r").shape[0]
        if len(saved_ids) != rows:
            reasons.append(f"ids ({len(saved_ids)}) and vectors ({rows}) disagree")
        elif saved_ids != [i["id"] for i in corpus]:
            reasons.append("saved ids do not match the corpus order")

    return not reasons, reasons


is_cached, reasons = cache_status(run_metadata)
print("cache usable:", is_cached)
for reason in reasons:
    print("  -", reason)


cache usable: False
  - no vectors file at eu_ai_act_embeddings.npy
  - no ids file at embedding_ids.json
  - no metadata file at embedding_metadata.json


## 5. Embed, or load from cache

This is the step the caching exists for. On a cold run the model is downloaded
(~130 MB) and 901 items are encoded; on every later run the vectors are read
straight from disk.


In [5]:
_model: SentenceTransformer | None = None


def get_model() -> SentenceTransformer:
    """Load the model once per session and reuse it."""
    global _model
    if _model is None:
        _model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    return _model


def compute_embeddings(items: list[dict], metadata: dict):
    """Encode every item and write vectors, ids and metadata to disk."""
    started = time.time()
    model = get_model()

    vectors = model.encode(
        [item[EMBED_FIELD] for item in items],
        batch_size=BATCH_SIZE,
        normalize_embeddings=NORMALIZE,
        convert_to_numpy=True,
        show_progress_bar=True,
    ).astype(DTYPE)

    ids = [item["id"] for item in items]
    metadata = {
        **metadata,
        "dimension": int(vectors.shape[1]),
        "max_seq_length": int(model.max_seq_length),
        "encode_seconds": round(time.time() - started, 1),
        "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

    EMBED_DIR.mkdir(parents=True, exist_ok=True)
    np.save(VECTORS_PATH, vectors)
    IDS_PATH.write_text(json.dumps(ids, ensure_ascii=False, indent=2), encoding="utf-8")
    METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2),
                             encoding="utf-8")
    return vectors, ids, metadata


if is_cached:
    embeddings = np.load(VECTORS_PATH)
    embedding_ids = json.loads(IDS_PATH.read_text(encoding="utf-8"))
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    print(f"Loaded cached embeddings, built {metadata.get('created_utc', 'unknown')}")
else:
    print("Rebuilding embeddings ...")
    embeddings, embedding_ids, metadata = compute_embeddings(corpus, run_metadata)
    print(f"Encoded in {metadata['encode_seconds']}s")

print(f"\nvectors {embeddings.shape} {embeddings.dtype}"
      f"  ({embeddings.nbytes / 1024 / 1024:.1f} MB)")


Rebuilding embeddings ...


Batches: 100%|██████████| 31/31 [00:21<00:00,  1.46it/s]

Encoded in 24.9s

vectors (980, 384) float32  (1.4 MB)


## 6. Verification

Cheap assertions, run every time. A silently misaligned index is the worst failure
mode in a RAG system: it returns confident answers citing the wrong provision.


In [6]:
assert embeddings.shape[0] == len(corpus), "one vector per item"
assert embeddings.shape[0] == len(embedding_ids), "one id per vector"
assert embedding_ids == [i["id"] for i in corpus], "ids align with corpus order"
assert embeddings.dtype == np.dtype(DTYPE), f"vectors should be {DTYPE}"
assert np.isfinite(embeddings).all(), "no NaN or inf values"

norms = np.linalg.norm(embeddings, axis=1)
if NORMALIZE:
    assert np.allclose(norms, 1.0, atol=1e-5), "vectors should be unit length"

# Index from item id to row, for looking up a provision's vector directly.
row_of = {item_id: row for row, item_id in enumerate(embedding_ids)}

print("all checks passed")
print(f"  items      {len(corpus)}")
print(f"  dimension  {embeddings.shape[1]}")
print(f"  L2 norms   min {norms.min():.6f}  max {norms.max():.6f}")
print(f"  model      {metadata['model_name']}")
print(f"  built      {metadata.get('created_utc', 'unknown')}")


all checks passed
  items      980
  dimension  384
  L2 norms   min 1.000000  max 1.000000
  model      BAAI/bge-small-en-v1.5
  built      2026-09-04T19:20:21+00:00


## 7. Sequence length

`bge-small-en-v1.5` truncates at **512 tokens**. Anything longer is silently cut off,
so it is worth knowing exactly which provisions are affected rather than discovering
it through a bad answer later.


In [7]:
# The model carries its own tokenizer, so this needs no extra download.
tokenizer = get_model().tokenizer
token_counts = np.array([len(tokenizer.encode(i[EMBED_FIELD])) for i in corpus])
limit = metadata.get("max_seq_length", 512)
over = token_counts > limit

print(f"token limit: {limit}")
for pct in (50, 90, 99, 100):
    print(f"  p{pct:<4} {int(np.percentile(token_counts, pct)):5} tokens")
print(f"\ntruncated items: {over.sum()} of {len(corpus)} ({100 * over.mean():.1f}%)")

for idx in np.argsort(-token_counts)[:5]:
    if over[idx]:
        print(f"  {token_counts[idx]:5} tokens  {corpus[idx]['id']:16} "
              f"{corpus[idx]['embed_text'][:46]}")


token limit: 512
  p50      93 tokens
  p90     248 tokens
  p99     493 tokens
  p100    512 tokens

truncated items: 0 of 980 (0.0%)


## 8. Sanity check — a similarity search

Vectors are normalised, so cosine similarity is just a dot product. Note the
`QUERY_PREFIX`: BGE expects the instruction on the query only, never on the passages.


In [8]:
def search(question: str, k: int = 5) -> list[tuple[float, dict]]:
    """Return the k items most similar to the question."""
    query = get_model().encode([QUERY_PREFIX + question],
                         normalize_embeddings=NORMALIZE,
                         convert_to_numpy=True).astype(DTYPE)
    scores = embeddings @ query[0]
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), corpus[i]) for i in top]


for question in ("Can I use AI to screen job applicants?",
                 "Is emotion recognition in the workplace allowed?"):
    print(f"\n=== {question} ===")
    for score, item in search(question):
        label = item.get("article_title") or item.get("annex_title", "")
        print(f"  {score:.3f}  {item['id']:18} {label[:52]}")



=== Can I use AI to screen job applicants? ===
  0.720  anx_III.point_4    High-risk AI systems referred to in Article 6(2)
  0.702  rct_57             
  0.676  rct_56             
  0.665  rct_59.chunk_2     
  0.659  art_2.para_8       Scope

=== Is emotion recognition in the workplace allowed? ===
  0.740  rct_44             
  0.736  art_50.para_3      Transparency obligations for providers and deployers
  0.727  rct_18             
  0.701  art_3.def_39       Definitions
  0.680  rct_57             


## Next steps

The retrieval index is in place. What follows:

1. **Answer generation** — feed the retrieved provisions to Claude with a prompt that
   requires every claim to cite an item id, and refuses to answer beyond the corpus.
2. **The question loop** — the interface itself, taking a plain-language question about
   an AI project and returning a grounded answer with citations.
3. **Handling the 21 truncated items** — either chunk them, or accept the truncation
   knowingly. `art_5.para_1` (prohibited practices) is the one that matters most.
